In [1]:
from pathlib import Path
import os
import sys
import time

import torch
import hydra
from hydra import compose
from hydra.core.global_hydra import GlobalHydra
from hydra.utils import instantiate
from omegaconf import DictConfig, OmegaConf
from torch.utils.data import DataLoader

PROJECT_ROOT = Path(
    "/homes/cmininno/tesi/surgical_scene_segmentation"
)

CONFIG_DIR = PROJECT_ROOT / "configs"
CONFIG_NAME = "config"

# Questa lista corrisponde agli override della riga Slurm/CLI.
#
# Esempio:
# python start_experiment.py \
#   dataset=baseline/cholec_seg8k \
#   training=baseline/cholec_seg8k \
#   model=baseline/mask2former \
#   train.optimizer.lr=1e-3

OVERRIDES = [
    "dataset=baseline/dsad",
    "training=baseline/dsad",
    "model=baseline/mask2former",
    "train.optimizer.lr=1e-3",

    # Debug locale: evita di aspettare 50 epoche per scoprire un bug.
    "train.max_epochs=1",
    "logging.wandb.enabled=false",
]

# True = esegue solo la costruzione + sanity check della pipeline.
# False = esegue anche Trainer.fit().
SANITY_CHECK_ONLY = False

# Se vuoi eseguire davvero il training dal notebook:
# SANITY_CHECK_ONLY = False

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project: {PROJECT_ROOT}")
print(f"Config : {CONFIG_DIR}")
print("Overrides:")
for override in OVERRIDES:
    print(f"  - {override}")


Project: /homes/cmininno/tesi/surgical_scene_segmentation
Config : /homes/cmininno/tesi/surgical_scene_segmentation/configs
Overrides:
  - dataset=baseline/dsad
  - training=baseline/dsad
  - model=baseline/mask2former
  - train.optimizer.lr=1e-3
  - train.max_epochs=1
  - logging.wandb.enabled=false


In [2]:
# ============================================================
# 1. Hydra: stessa composizione usata dal job
# ============================================================

GlobalHydra.instance().clear()

with hydra.initialize_config_dir(
    version_base=None,
    config_dir=str(CONFIG_DIR),
):
    cfg = compose(
        config_name=CONFIG_NAME,
        overrides=OVERRIDES,
    )

print(OmegaConf.to_yaml(cfg, resolve=False))


name: dsad
data:
  name: dsad
  dataset:
    _target_: src.datasets.dsad_dataset.DSADDataset
    root: ${paths.data}/DSAD
    variant: multi_organ
  num_classes: ${data.class_config.${data.dataset.variant}}
  class_config:
    single_organ: 2
    multi_organ: 8
  split:
    strategy: official
    group_by: surgery
    stratified: false
preprocessing:
  image:
    size:
    - 384
    - 480
    channel_first: true
    dtype: float32
    normalize: true
  mask:
    channel_first: true
    dtype: long
augmentation:
  train:
    enabled: true
    scaling:
      enabled: true
      range:
      - -0.1
      - 0.1
      probability: 0.5
    rotation:
      enabled: true
      range:
      - -15
      - 15
      probability: 0.5
    brightness:
      enabled: true
      probability: 0.5
    contrast:
      enabled: true
      probability: 0.5
  validation:
    enabled: false
  test:
    enabled: false
train:
  batch_size: 4
  accumulation_steps: 1
  max_epochs: 1
  eval_freq: 1
  num_workers: 

In [3]:
# ============================================================
# 2. Setup
# ============================================================

from src.utils.repro import set_seed, seed_worker

experiment_start = time.time()

set_seed(cfg.experiment.seed)

generator = torch.Generator()
generator.manual_seed(cfg.experiment.seed)

device = torch.device(
    cfg.device if torch.cuda.is_available() else "cpu"
)

print(f"Seed   : {cfg.experiment.seed}")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")


Seed   : 42
Device : cuda
GPU    : Quadro RTX 5000


In [4]:
# ============================================================
# 3. Dataset
# ============================================================

from src.datasets.transformation.factory import TransformFactory
from src.datasets.transformation.transform_dataset import TransformDataset

print(f"Dataset: {cfg.data.name}")

dataset = instantiate(cfg.data.dataset)

print(f"Dataset class : {dataset.__class__.__name__}")
print(f"Dataset size  : {len(dataset)}")
print(f"Num classes   : {dataset.num_classes}")
print(f"Class names   : {dataset.CLASS_NAMES}")


/homes/cmininno/tesi/tesi_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset: dsad
Dataset class : DSADDataset
Dataset size  : 1430
Num classes   : 8
Class names   : ['background', 'abdominal_wall', 'colon', 'liver', 'pancreas', 'small_intestine', 'spleen', 'stomach']


In [5]:
# ============================================================
# 4. Dataset split
# ============================================================

split_cfg = cfg.data.split

split_kwargs = {
    "strategy": split_cfg.strategy,
    "group_by": split_cfg.group_by,
}

if split_cfg.strategy == "percentage":
    split_kwargs.update(
        {
            "train_ratio": split_cfg.train_ratio,
            "val_ratio": split_cfg.val_ratio,
            "test_ratio": split_cfg.test_ratio,
            "seed": split_cfg.get(
                "seed",
                cfg.experiment.seed,
            ),
            "stratified": split_cfg.get(
                "stratified",
                False,
            ),
        }
    )

print("Split configuration:")
print(OmegaConf.to_yaml(split_cfg))

train_dataset, val_dataset, test_dataset = dataset.create_split(
    **split_kwargs
)

print()
print(f"Train samples: {len(train_dataset) if train_dataset else 0}")
print(f"Val samples  : {len(val_dataset) if val_dataset else 0}")
print(f"Test samples : {len(test_dataset) if test_dataset else 0}")


Split configuration:
strategy: official
group_by: surgery
stratified: false


DATASET SPLIT

Train:
  Groups : ['04', '05', '08', '10', '12', '16', '17', '22', '23', '24', '25', '27', '29', '30', '31']
  Frames : 863

Validation:
  Groups : ['03', '21', '26']
  Frames : 202

Test:
  Groups : ['02', '07', '11', '18', '20']
  Frames : 365

Total:
  Groups : 23
  Frames : 1430

Train samples: 863
Val samples  : 202
Test samples : 365


In [6]:
# ============================================================
# 5. Transforms
# ============================================================

transform_factory = TransformFactory(cfg)

train_transform = transform_factory.build_train_transform()
val_transform = transform_factory.build_val_transform()

test_transform = (
    transform_factory.build_test_transform()
    if test_dataset is not None
    else None
)

if train_dataset is not None:
    train_dataset = TransformDataset(
        train_dataset,
        transform=train_transform,
    )

if val_dataset is not None:
    val_dataset = TransformDataset(
        val_dataset,
        transform=val_transform,
    )

if test_dataset is not None:
    test_dataset = TransformDataset(
        test_dataset,
        transform=test_transform,
    )

print("Transforms and TransformDataset wrappers created.")


Transforms and TransformDataset wrappers created.


In [7]:
# ============================================================
# 6. DataLoaders
# ============================================================

train_loader = None
val_loader = None
test_loader = None

if train_dataset is not None:
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.train.batch_size,
        shuffle=True,
        num_workers=cfg.train.num_workers,
        pin_memory=cfg.train.pin_memory,
        worker_init_fn=seed_worker,
        generator=generator,
    )

if val_dataset is not None:
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.train.batch_size,
        shuffle=False,
        num_workers=cfg.train.num_workers,
        pin_memory=cfg.train.pin_memory,
        worker_init_fn=seed_worker,
        generator=generator,
    )

if test_dataset is not None:
    test_loader = DataLoader(
        test_dataset,
        batch_size=cfg.train.batch_size,
        shuffle=False,
        num_workers=cfg.train.num_workers,
        pin_memory=cfg.train.pin_memory,
        worker_init_fn=seed_worker,
        generator=generator,
    )

print(f"Train batches: {len(train_loader) if train_loader else 0}")
print(f"Val batches  : {len(val_loader) if val_loader else 0}")
print(f"Test batches : {len(test_loader) if test_loader else 0}")


Train batches: 216
Val batches  : 51
Test batches : 92


In [8]:
# ============================================================
# 7. Inspect one transformed batch
# ============================================================

batch = next(iter(val_loader))

print("Batch keys:")
print(list(batch.keys()))

for key, value in batch.items():
    if torch.is_tensor(value):
        print(
            f"{key:>8}: shape={tuple(value.shape)}, "
            f"dtype={value.dtype}"
        )
    else:
        print(f"{key:>8}: type={type(value).__name__}")


Batch keys:
['image', 'mask', 'surgery_id', 'frame_id']
   image: shape=(4, 3, 384, 480), dtype=torch.float32
    mask: shape=(4, 1, 384, 480), dtype=torch.int64
surgery_id: type=list
frame_id: type=list


In [9]:
from src.models.mask2former_wrapper import Mask2FormerWrapper

In [10]:
# ============================================================
# 8. Model
# ============================================================

#model = instantiate(cfg.model.architecture).to(device)


num_classes = cfg.data.num_classes

model = Mask2FormerWrapper(
    checkpoint=cfg.model.architecture.checkpoint,
    num_labels=num_classes,
)

print(f"Model class: {model.__class__.__name__}")
print(f"Mask2Former: {getattr(model, 'is_mask2former', False)}")

num_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(f"Parameters         : {num_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")


/homes/cmininno/tesi/tesi_venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of Mask2FormerForUniversalSegmentation were not initialized from the model checkpoint at facebook/mask2former-swin-base-ade-semantic and are newly initialized because the shapes did not match:
- class_predictor.bias: found shape torch.Size([151]) in the checkpoint and torch.Size([9]) in the model instantiated
- class_predictor.weight: found shape torch.Size([151, 256]) in the checkpoint and torch.Size([9, 256]) in the model instantiated
- criterion.empty_weight: found shape torch.Size([151]) in the checkpoint and torch.Size([9]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model class: Mask2FormerWrapper
Mask2Former: True
Parameters         : 106,885,697
Trainable parameters: 106,885,697


In [11]:
# ============================================================
# 9. Optimizer, scheduler, loss, scaler
# ============================================================

optimizer = instantiate(
    cfg.train.optimizer,
    params=model.parameters(),
)

scheduler = instantiate(
    cfg.train.scheduler,
    optimizer=optimizer,
)

loss_fn = instantiate(cfg.train.loss)

scaler = (
    instantiate(cfg.train.scaler)
    if "scaler" in cfg.train
    else None
)

print("Optimizer :", optimizer.__class__.__name__)
print("Scheduler :", scheduler.__class__.__name__)
print("Loss      :", loss_fn.__class__.__name__)
print("Scaler    :", scaler.__class__.__name__ if scaler else None)
print("LR        :", optimizer.param_groups[0]["lr"])


Optimizer : AdamW
Scheduler : CosineAnnealingLR
Loss      : DiceFocalLoss
Scaler    : GradScaler
LR        : 0.001


In [12]:
# ============================================================
# 10. Metric manager
# ============================================================

from src.metrics.metric_manager import MetricManager

metrics = {
    name: instantiate(metric_cfg)
    for name, metric_cfg in cfg.evaluation.metrics.items()
}

class_names = {
    index: name
    for index, name in enumerate(dataset.CLASS_NAMES)
}

metric_manager = MetricManager(
    metrics=metrics,
    num_classes=dataset.num_classes,
    dataset_name=cfg.data.name,
    class_names=class_names,
)

print("Metrics:", metric_manager.names)


Metrics: ['dice', 'iou', 'hd95']


In [13]:
# ============================================================
# 11. Sanity check: forward + shape contract
# ============================================================

def describe_tensor(name, tensor):
    if tensor is None:
        print(f"{name}: None")
        return

    print(
        f"{name}: shape={tuple(tensor.shape)}, "
        f"dtype={tensor.dtype}, "
        f"device={tensor.device}"
    )


def check_segmentation_shape(
    predictions,
    masks,
    num_classes,
):
    if predictions is None:
        raise RuntimeError(
            "Model returned None instead of segmentation predictions."
        )

    if predictions.ndim != 4:
        raise ValueError(
            "Expected predictions with shape [B, C, H, W], "
            f"got {tuple(predictions.shape)}"
        )

    if masks.ndim not in (3, 4):
        raise ValueError(
            f"Unexpected target shape: {tuple(masks.shape)}"
        )

    # --------------------------------------------------------
    # Batch size
    # --------------------------------------------------------

    if predictions.shape[0] != masks.shape[0]:
        raise ValueError(
            "Batch size mismatch:\n"
            f"  predictions: {tuple(predictions.shape)}\n"
            f"  masks      : {tuple(masks.shape)}"
        )

    # --------------------------------------------------------
    # Number of classes
    # --------------------------------------------------------

    predicted_classes = predictions.shape[1]

    if predicted_classes != num_classes:
        raise ValueError(
            "CLASS COUNT MISMATCH:\n"
            f"  predictions: {predicted_classes}\n"
            f"  config     : {num_classes}"
        )

    # --------------------------------------------------------
    # Spatial dimensions
    # --------------------------------------------------------

    target_hw = tuple(masks.shape[-2:])
    prediction_hw = tuple(predictions.shape[-2:])

    if prediction_hw != target_hw:
        raise ValueError(
            "SEGMENTATION SHAPE MISMATCH:\n"
            f"  predictions : {tuple(predictions.shape)}\n"
            f"  target      : {tuple(masks.shape)}\n"
            f"  prediction spatial size = {prediction_hw}\n"
            f"  target spatial size     = {target_hw}"
        )


# ============================================================
# Batch
# ============================================================

images = batch["image"].to(device)
masks = batch["mask"].to(device)

num_classes = int(cfg.data.num_classes)

describe_tensor("images", images)
describe_tensor("masks", masks)


# ============================================================
# Inference forward
# ============================================================

model.eval()

with torch.no_grad():
    predictions, loss = model(images)

describe_tensor("predictions", predictions)
describe_tensor("loss", loss)


# ============================================================
# Shape validation
# ============================================================

check_segmentation_shape(
    predictions=predictions,
    masks=masks,
    num_classes=num_classes,
)

print("Inference shape sanity check: PASSED")

images: shape=(4, 3, 384, 480), dtype=torch.float32, device=cuda:0
masks: shape=(4, 1, 384, 480), dtype=torch.int64, device=cuda:0


RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

In [ ]:
# ============================================================
# 12. Optional direct metric sanity check
# ============================================================

# Questo passaggio è volutamente separato dal training.
# Se fallisce, non ha senso lanciare 50 epoche su Slurm.

metric_manager.reset()

metric_manager.update(
    predictions.detach(),
    masks.detach(),
)

metric_preview = metric_manager.compute()

print("Metric sanity check: PASSED")

for name, value in metric_preview.items():
    print(f"{name:>10}: {value:.6f}")

In [ ]:
#from __future__ import annotations
#
#import time
#from pathlib import Path
#from typing import Any
#
#import numpy as np
#import torch
#from tqdm import tqdm
#
#
#class Trainer:
#
#    def __init__(
#        self,
#        model: torch.nn.Module,
#        train_loader: torch.utils.data.DataLoader,
#        val_loader: torch.utils.data.DataLoader,
#        optimizer: torch.optim.Optimizer,
#        scheduler: Any,
#        loss_fn: torch.nn.Module,
#        scaler: Any,
#        metric_manager: Any,
#        cfg: Any,
#        device: torch.device,
#        logger: Any | None = None,
#    ) -> None:
#
#        self.model = model.to(device)
#        self.train_loader = train_loader
#        self.val_loader = val_loader
#        self.optimizer = optimizer
#        self.scheduler = scheduler
#        self.loss_fn = loss_fn
#        self.scaler = scaler
#        self.metric_manager = metric_manager
#        self.cfg = cfg
#        self.device = device
#        self.logger = logger
#
#        train_cfg = cfg.train
#        self.max_epochs = int(train_cfg.get("max_epochs", 50))
#        self.accumulation_steps = max(1, int(train_cfg.get("accumulation_steps", 1)))
#        self.eval_freq = max(1, int(train_cfg.get("eval_freq", 1)))
#
#        logging_cfg = cfg.get("logging", {})
#        qualitative_cfg = logging_cfg.get("qualitative", {})
#        self.qualitative_enabled = bool(qualitative_cfg.get("enabled", True))
#        self.qualitative_freq = max(1, int(qualitative_cfg.get("frequency", 1)))
#        self.qualitative_max_samples = max(1, int(qualitative_cfg.get("max_samples", 4)))
#
#        early_stopping_cfg = train_cfg.get("early_stopping", {})
#        self.early_stopping_enabled = bool(early_stopping_cfg.get("enabled", True))
#        self.patience = int(early_stopping_cfg.get("patience", 10))
#        self.monitor = str(early_stopping_cfg.get("monitor", "dice"))
#        self.monitor_mode = str(early_stopping_cfg.get("mode", "max")).lower()
#
#        if self.monitor_mode not in {"min", "max"}:
#            raise ValueError("early_stopping.mode must be either 'min' or 'max'")
#
#        self.metric_names = list(self.metric_manager.names)
#        available_metrics = set(self.metric_names) | {"loss", "fps"}
#
#        if self.monitor not in available_metrics:
#            raise ValueError(
#                f"Early stopping monitor '{self.monitor}' is not available. "
#                f"Available metrics: {sorted(available_metrics)}"
#            )
#
#        self.best_score = float("inf") if self.monitor_mode == "min" else -float("inf")
#        self.best_epoch = 0
#        self.best_metrics: dict[str, float] = {}
#        self.epochs_without_improvement = 0
#
#        self.output_dir = Path(cfg.get("output_dir", "./outputs"))
#        self.output_dir.mkdir(parents=True, exist_ok=True)
#
#        lr = float(self.optimizer.param_groups[0]["lr"])
#        lr_str = f"{lr:.0e}".replace("e-0", "e-").replace("e+0", "e+")
#
#        self.best_checkpoint_path = (
#            self.output_dir / f"best_{self.cfg.data.name}_{self.cfg.model.name}_lr_{lr_str}.pt"
#        )
#
#    # Forward
#    def _forward(
#        self,
#        images: torch.Tensor,
#        masks: torch.Tensor | None = None,
#    ) -> tuple[torch.Tensor, torch.Tensor | None]:
#        
#        return self.model(images, masks)
#
#    # Batch preparation
#    def _prepare_batch(
#        self,
#        batch: dict[str, Any],
#    ) -> tuple[torch.Tensor, torch.Tensor]:
#    
#        images = batch["image"].to(self.device, non_blocking=True)
#        masks = batch["mask"].to(self.device, non_blocking=True)
#
#        # Semantic segmentation masks:
#        # [B, H, W] -> [B, 1, H, W]
#        if masks.ndim == 3:
#            masks = masks.unsqueeze(1)
#
#        if masks.ndim != 4:
#            raise ValueError(
#                f"Invalid mask shape: {tuple(masks.shape)}. "
#                "Expected [B, 1, H, W] for multiclass semantic segmentation."
#            )
#
#        if masks.shape[1] != 1:
#            raise ValueError(
#                f"Invalid mask channel dimension: {tuple(masks.shape)}. "
#                "Expected [B, 1, H, W], not one-hot [B, C, H, W]."
#            )
#
#        return images, masks
#
#    # AMP
#    def _autocast(self):
#        precision = str(self.cfg.train.get("precision", "32"))
#        enabled = precision == "16-mixed" and self.device.type == "cuda"
#        return torch.amp.autocast(device_type="cuda", enabled=enabled)
#
#    def _optimizer_step(self) -> None:
#        if self.scaler is not None:
#            self.scaler.unscale_(self.optimizer)
#
#        torch.nn.utils.clip_grad_norm_(
#            self.model.parameters(),
#            max_norm=1.0,
#        )
#
#        if self.scaler is not None:
#            self.scaler.step(self.optimizer)
#            self.scaler.update()
#        else:
#            self.optimizer.step()
#
#        self.optimizer.zero_grad(set_to_none=True)
#
#    # Training
#    def train_epoch(self, epoch: int) -> tuple[float, dict[str, float], dict[str, float]]:
#        
#        self.model.train()
#        self.metric_manager.reset()
#
#        self.optimizer.zero_grad(set_to_none=True)
#
#        running_loss = 0.0
#        num_batches = 0
#
#        progress = tqdm(
#            self.train_loader,
#            desc=f"Epoch {epoch} [Train]",
#            leave=False,
#        )
#
#        for step, batch in enumerate(progress):
#            
#            images, masks = self._prepare_batch(batch)
#
#            with self._autocast():
#                predictions, loss = self._forward(images, masks)
#
#            if loss is None:
#                raise RuntimeError("The model returned no loss during training.")
#
#            loss_for_backward = loss / self.accumulation_steps
#
#            if self.scaler is not None:
#                self.scaler.scale(loss_for_backward).backward()
#            else:
#                loss_for_backward.backward()
#
#            if (step + 1) % self.accumulation_steps == 0 or (step + 1) == len(self.train_loader):
#                self._optimizer_step()
#
#            self.metric_manager.update(predictions.detach(), masks.detach())
#
#            running_loss += loss.detach().item()
#            num_batches += 1
#
#            progress.set_postfix(loss=f"{loss.item():.4f}")
#
#        metrics = self.metric_manager.compute()
#        loss_value = running_loss / max(num_batches, 1)
#        per_class = self._extract_per_class_metrics(metrics)
#
#        return loss_value, metrics, per_class
#
#
#    # Validation
#    @torch.no_grad()
#    def validate(
#        self, epoch: int
#    ) -> tuple[float, dict[str, float], dict[str, float], dict[str, torch.Tensor] | None]:
#        
#        self.model.eval()
#        self.metric_manager.reset()
#
#        running_loss = 0.0
#        num_batches = 0
#        qualitative_batch = None
#
#        progress = tqdm(
#            self.val_loader,
#            desc=f"Epoch {epoch} [Val]",
#            leave=False,
#        )
#
#        for batch_index, batch in enumerate(progress):
#            images, masks = self._prepare_batch(batch)
#
#            with self._autocast():
#                predictions, model_loss = self._forward(images, masks)
#
#            if model_loss is None:
#                if self.loss_fn is None:
#                    raise RuntimeError("Model returned no loss and no loss function is configured.")
#                loss = self.loss_fn(predictions, masks)
#            else:
#                loss = model_loss
#
#            self.metric_manager.update(predictions.detach(), masks.detach())
#
#            running_loss += loss.detach().item()
#            num_batches += 1
#
#            if (
#                batch_index == 0
#                and self.qualitative_enabled
#                and epoch % self.qualitative_frequency == 0
#            ):
#                n_samples = min(self.qualitative_max_samples, images.shape[0])
#                
#                prediction = predictions[:n_samples]
#
#                if prediction.ndim == 4:
#                    prediction = prediction.argmax(dim=1)
#
#                qualitative_batch = {
#                    "image": images[:n_samples].detach().cpu(),
#                    "mask": masks[:n_samples].detach().cpu(),
#                    "prediction": self._prediction_to_mask(predictions[:n_samples]),
#                }
#
#            progress.set_postfix(loss=f"{loss.item():.4f}")
#
#        metrics = self.metric_manager.compute()
#        loss_value = running_loss / max(num_batches, 1)
#        per_class = self._extract_per_class_metrics(metrics)
#
#        return loss_value, metrics, per_class, qualitative_batch
#
#    # Checkpoint
#    def _save_checkpoint(
#        self,
#        path: Path,
#        epoch: int,
#        metrics: dict[str, float],
#    ) -> None:
#        
#        checkpoint = {
#            "epoch": epoch,
#            "model_state_dict": self.model.state_dict(),
#            "optimizer_state_dict": self.optimizer.state_dict(),
#            "scheduler_state_dict": self.scheduler.state_dict() if self.scheduler is not None else None,
#            "best_score": self.best_score,
#            "best_epoch": self.best_epoch,
#            "best_metrics": self.best_metrics,
#            "metrics": metrics,
#            "config": self.cfg,
#        }
#
#        if self.scaler is not None:
#            checkpoint["scaler_state_dict"] = self.scaler.state_dict()
#
#        if self.logger is not None:
#
#            lr = float(self.optimizer.param_groups[0]["lr"])
#            lr_str = f"{lr:.0e}"
#
#            self.logger.log_checkpoint(
#                checkpoint=checkpoint,
#                checkpoint_path=path,
#                name=f"model_lr_{lr_str}",
#                aliases=[f"epoch-{epoch}", "latest"],
#            )
#
#        else:
#            path.parent.mkdir(parents=True, exist_ok=True)
#            torch.save(checkpoint, path)
#
#
#    # Early stopping
#    def _is_improvement(self, score: float) -> bool:
#        if self.monitor_mode == "min":
#            return score < self.best_score
#        return score > self.best_score
#
#    def _update_best(
#        self,
#        epoch: int,
#        metrics: dict[str, float],
#    ) -> bool:
#
#        if self.monitor not in metrics:
#            raise KeyError(
#                f"Monitor metric '{self.monitor}' not found in validation metrics: "
#                f"{list(metrics.keys())}"
#            )
#
#        score = float(metrics[self.monitor])
#
#        if not np.isfinite(score) or not self._is_improvement(score):
#            return False
#
#        self.best_score = score
#        self.best_epoch = epoch
#        self.best_metrics = dict(metrics)
#
#        self._save_checkpoint(self.best_checkpoint_path, epoch, metrics)
#        self.epochs_without_improvement = 0
#
#        return True
#
#    # Logging
#    def _log_epoch(
#        self,
#        epoch: int,
#        train_metrics: dict[str, float],
#        val_metrics: dict[str, float],
#        per_class: dict[str, float],
#        qualitative_batch: dict[str, torch.Tensor] | None,
#    ) -> None:
#
#        if self.logger is None:
#            return
#
#        self.logger.log_epoch(
#            epoch=epoch,
#            train_metrics=train_metrics,
#            val_metrics=val_metrics,
#            per_class_metrics=per_class,
#            learning_rate=self.optimizer.param_groups[0]["lr"],
#            qualitative_batch=qualitative_batch,
#            class_labels=self.class_labels,
#            qualitative_frequency=self.qualitative_freq,
#        )
#
#    # Training loop
#    def fit(self) -> dict[str, Any]:
#        
#        history: list[dict[str, Any]] = []
#
#        for epoch in range(1, self.max_epochs + 1):
#
#            epoch_start = time.perf_counter()
#
#            # Train
#            train_loss, train_metrics, train_per_class = self.train_epoch(epoch)
#            train_metrics["loss"] = train_loss
#
#            # Validation
#            if epoch % self.eval_freq == 0:
#                val_loss, val_metrics, per_class_results, qualitative_batch = self.validate(epoch)
#                val_metrics["loss"] = val_loss
#
#                self._log_epoch(
#                    epoch=epoch,
#                    train_metrics=train_metrics,
#                    val_metrics=val_metrics,
#                    per_class_results=per_class_results,
#                    qualitative_batch=qualitative_batch,
#                )
#
#                improved = self._update_best(epoch, val_metrics)
#
#                if not improved:
#                    self.epochs_without_improvement += 1
#
#                epoch_time = time.perf_counter() - epoch_start
#                epoch_results = {
#                    "epoch": epoch,
#                    "train": train_metrics,
#                    "val": val_metrics,
#                    "per_class": per_class_results,
#                    "epoch_time": epoch_time,
#                }
#                history.append(epoch_results)
#
#                print(
#                    f"[Epoch {epoch:03d}] "
#                    f"train_loss={train_metrics.get('loss', float('nan')):.4f} "
#                    f"val_loss={val_metrics.get('loss', float('nan')):.4f} "
#                    f"{self.monitor}={val_metrics.get(self.monitor, float('nan')):.4f}"
#                )
#
#                if improved:
#                    print(f"  -> New best checkpoint ({self.monitor}={self.best_score:.4f})")
#
#                # Early stopping
#                if self.early_stopping_enabled and self.epochs_without_improvement >= self.patience:
#                    print("Early stopping triggered.")
#                    break
#
#            else:
#                epoch_time = time.perf_counter() - epoch_start
#                epoch_results = {
#                    "epoch": epoch,
#                    "train": train_metrics,
#                    "epoch_time": epoch_time,
#                }
#                history.append(epoch_results)
#
#            if self.scheduler is not None:
#                self.scheduler.step()
#
#        return {
#            "history": history,
#            "best_epoch": self.best_epoch,
#            "best_score": self.best_score,
#            "best_metrics": self.best_metrics,
#            "best_checkpoint": str(self.best_checkpoint_path),
#        }

In [19]:
# ============================================================
# 13. Trainer
# ============================================================

from src.engine.trainer import Trainer

# Per replicare completamente start_experiment.py, abilita WandB
# e inizializza il logger nella cella successiva.
#
# Per il debug locale è più semplice usare logger=None.

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    loss_fn=loss_fn,
    scaler=scaler,
    metric_manager=metric_manager,
    cfg=cfg,
    device=device,
    logger=None,
)

print("Trainer initialized.")


Trainer initialized.


In [18]:
import importlib

import src.engine.trainer as trainer

importlib.reload(trainer)

Trainer = trainer.Trainer

In [ ]:
print(torch.cuda.memory_summary(device=device, abbreviated=True))

In [21]:
# ============================================================
# 14. Training
# ============================================================

if SANITY_CHECK_ONLY:
    print(
        "SANITY_CHECK_ONLY=True: training skipped. "
        "Set it to False to execute Trainer.fit()."
    )
    train_results = None
else:
    train_results = trainer.fit()
    print("\nTraining completed.")
    print("Best epoch :", train_results["best_epoch"])
    print("Best score :", train_results["best_score"])
    print(
        "Best checkpoint:",
        train_results["best_checkpoint"],
    )


OutOfMemoryError: CUDA out of memory. Tried to allocate 52.00 MiB. GPU 0 has a total capacty of 15.74 GiB of which 41.50 MiB is free. Including non-PyTorch memory, this process has 8.43 GiB memory in use. Process 404099 has 2.12 GiB memory in use. Process 404765 has 5.15 GiB memory in use. Of the allocated memory 8.04 GiB is allocated by PyTorch, and 194.94 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [ ]:
# ============================================================
# 15. Load best checkpoint
# ============================================================

if train_results is not None:
    checkpoint_path = train_results["best_checkpoint"]

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    print(f"Loaded best checkpoint: {checkpoint_path}")


In [ ]:
# ============================================================
# 15. Load best checkpoint
# ============================================================

if train_results is not None:
    checkpoint_path = train_results["best_checkpoint"]

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    print(f"Loaded best checkpoint: {checkpoint_path}")


In [ ]:
# ============================================================
# 16. Optional final test
# ============================================================

if train_results is not None and test_loader is not None:
    from src.engine.tester import Tester

    test_metrics = {
        name: instantiate(metric_cfg)
        for name, metric_cfg in cfg.evaluation.metrics.items()
    }

    test_metric_manager = MetricManager(
        metrics=test_metrics,
        num_classes=dataset.num_classes,
        dataset_name=cfg.data.name,
        class_names=class_names,
    )

    tester = Tester(
        model=model,
        test_loader=test_loader,
        metric_manager=test_metric_manager,
        loss_fn=loss_fn,
        cfg=cfg,
        device=device,
        logger=None,
    )

    test_results = tester.test()

    print("Final test results:")
    for name, value in test_results.get("metrics", {}).items():
        print(f"{name:>10}: {value:.6f}")
elif train_results is not None:
    print("No test split available; final test skipped.")


In [ ]:
# ============================================================
# 17. Final summary
# ============================================================

elapsed = time.time() - experiment_start

print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"Dataset       : {cfg.data.name}")
print(f"Model         : {cfg.model.name}")
print(f"Device        : {device}")
print(f"Elapsed       : {elapsed:.2f}s")

if train_results is not None:
    print(f"Best epoch    : {train_results['best_epoch']}")
    print(f"Best score    : {train_results['best_score']}")
    print(
        f"Checkpoint    : {train_results['best_checkpoint']}"
    )
else:
    print("Training       : skipped")
